# Bronze Layer: Raw Data Ingestion

## Objective
The **Bronze Layer** is the first stage of our medallion architecture. It serves as the raw landing zone where data from external sources is ingested with minimal transformation. Our goal here is to:

1. **Load the raw CSV data** from the Kaggle Superstore dataset
2. **Create the bronze schema** in DuckDB to organize our data
3. **Verify data ingestion** by checking row counts and previewing the data

**Why Bronze?** At this stage, we preserve all data as-is, including any quality issues. This gives us a complete audit trail and allows us to diagnose problems in downstream layers.

---

## Step 1: Initialize DuckDB Connection & Create Schema

First, we establish a connection to our DuckDB database and create a dedicated schema for bronze-layer tables. Using schemas helps organize our data warehouse into logical layers.

In [5]:
import duckdb

# Connect to DuckDB database (creates if doesn't exist)
con = duckdb.connect("../data/retailion.duckdb")

# Create the bronze schema for raw data
con.execute("CREATE SCHEMA IF NOT EXISTS bronze;")

print("✅ Connected to DuckDB and created bronze schema")

✅ Connected to DuckDB and created bronze schema


## Step 2: Load Raw CSV Data

Now we ingest the raw superstore CSV file directly into the bronze schema. We use DuckDB's `read_csv_auto()` function which:
- **Automatically detects data types** (dates, numbers, strings)
- **Sets `ignore_errors=true`** to skip any malformed rows gracefully
- **Preserves column names** exactly as they appear in the CSV

This approach ensures we capture all usable data while being resilient to minor data quality issues.

In [6]:
# Load CSV data into the bronze layer
con.execute("""
CREATE OR REPLACE TABLE bronze.superstore AS 
SELECT * FROM read_csv_auto(
    '../data/Sample - Superstore.csv', 
    header=True,
    ignore_errors=True
);
""")

print("✅ CSV data loaded into bronze.superstore")

✅ CSV data loaded into bronze.superstore


## Step 3: Verify Data Ingestion

Let's validate that the data was loaded successfully by checking:
- **Total number of rows** ingested
- **Sample records** to visually inspect the data
- **Column structure** to understand what fields we're working with

In [7]:
# Get row count
count = con.execute("SELECT COUNT(*) FROM bronze.superstore").fetchone()[0]
print(f"✅ Bronze Layer successfully processed! Total raw data: {count:,} rows.\n")

# Display first 3 rows
display(con.execute("SELECT * FROM bronze.superstore LIMIT 3").df())

✅ Bronze Layer successfully processed! Total raw data: 9,627 rows.



,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


## Results Summary

✅ **Data Ingestion Complete!**

- **Total Records:** 9,627 rows of superstore transaction data
- **Total Columns:** 21 fields covering customers, products, locations, and financial metrics
- **Data Types:** Mix of text, dates, and numeric values (exact types determined by DuckDB's auto-detection)

## What's Next?

The bronze layer now contains our raw data. In the **Silver Layer** (notebook 02_silver.ipynb), we will:
1. Explore and audit data quality (nulls, duplicates, distributions)
2. Apply type casting and standardization
3. Clean the data for analytical use
4. Perform exploratory data analysis (EDA) with visualizations

In [8]:
# Close connection (optional, but good practice)
con.close()